# RetainIQ — Phase 2.2: Text & Categorical Standardization

## Objective

Implement the text and business-state transformations approved during Phase 1 while measuring
exactly what changed.

## 1. Load Raw Source

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("C:\\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\\RetainIQ_phase_02_data_cleaning_full\\data\\telco_raw.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("telco_raw.csv")

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded raw dataset: {df_raw.shape[0]:,} rows × {df_raw.shape[1]:,} columns")

Loaded raw dataset: 7,043 rows × 50 columns


## 2. Create an Isolated Working Copy

In [3]:
df_clean = df_raw.copy()
print(f"Working copy: {df_clean.shape[0]:,} rows × {df_clean.shape[1]:,} columns")

Working copy: 7,043 rows × 50 columns


## 3. Audit Leading/Trailing Whitespace Before Cleaning

In [4]:
string_cols = df_clean.select_dtypes(include="object").columns
whitespace_before = pd.Series({
    c: int((df_clean[c].notna() & (df_clean[c] != df_clean[c].str.strip())).sum())
    for c in string_cols
}, name="whitespace_issues").sort_values(ascending=False)

whitespace_before[whitespace_before > 0]

Series([], Name: whitespace_issues, dtype: int64)

### Interpretation

The supplied source currently has no detected leading/trailing whitespace issues. We still apply
defensive stripping because the cleaning pipeline should remain robust to future source refreshes.

## 4. Apply Defensive String Standardization

In [5]:
for c in string_cols:
    df_clean[c] = df_clean[c].str.strip()

whitespace_after = pd.Series({
    c: int((df_clean[c].notna() & (df_clean[c] != df_clean[c].str.strip())).sum())
    for c in string_cols
})

print(f"Whitespace-affected cells after cleaning: {int(whitespace_after.sum()):,}")

Whitespace-affected cells after cleaning: 0


In [6]:
assert int(whitespace_after.sum()) == 0
print("PASS — No leading/trailing whitespace remains.")

PASS — No leading/trailing whitespace remains.


## 5. Normalize `Offer`

In [7]:
offer_nulls_before = int(df_clean["Offer"].isna().sum())

df_clean["Offer"] = df_clean["Offer"].fillna("No Offer")

offer_nulls_after = int(df_clean["Offer"].isna().sum())

pd.Series({
    "Nulls Before": offer_nulls_before,
    "Nulls After": offer_nulls_after,
    "Rows Converted": offer_nulls_before
})

Nulls Before      3877
Nulls After          0
Rows Converted    3877
dtype: int64

In [8]:
assert offer_nulls_after == 0
assert int(df_clean["Offer"].eq("No Offer").sum()) == offer_nulls_before
print("PASS — Offer nulls normalized to 'No Offer'.")

PASS — Offer nulls normalized to 'No Offer'.


## 6. Normalize `Internet Type`

In [9]:
internet_nulls_before = int(df_clean["Internet Type"].isna().sum())

df_clean["Internet Type"] = df_clean["Internet Type"].fillna("No Internet Service")

internet_nulls_after = int(df_clean["Internet Type"].isna().sum())

pd.Series({
    "Nulls Before": internet_nulls_before,
    "Nulls After": internet_nulls_after,
    "Rows Converted": internet_nulls_before
})

Nulls Before      1526
Nulls After          0
Rows Converted    1526
dtype: int64

In [10]:
assert internet_nulls_after == 0
assert int(df_clean["Internet Type"].eq("No Internet Service").sum()) == internet_nulls_before
print("PASS — Internet Type nulls normalized to 'No Internet Service'.")

PASS — Internet Type nulls normalized to 'No Internet Service'.


## 7. Preserve Structural Churn Nulls

In [11]:
churn_category_before = int(df_raw["Churn Category"].isna().sum())
churn_reason_before = int(df_raw["Churn Reason"].isna().sum())

pd.DataFrame({
    "Field": ["Churn Category", "Churn Reason"],
    "Raw Nulls": [churn_category_before, churn_reason_before],
    "Clean Nulls": [
        int(df_clean["Churn Category"].isna().sum()),
        int(df_clean["Churn Reason"].isna().sum())
    ]
})

,Field,Raw Nulls,Clean Nulls
0,Churn Category,5174,5174
1,Churn Reason,5174,5174


In [12]:
assert df_clean["Churn Category"].isna().sum() == df_raw["Churn Category"].isna().sum()
assert df_clean["Churn Reason"].isna().sum() == df_raw["Churn Reason"].isna().sum()
print("PASS — Structural churn nulls preserved.")

PASS — Structural churn nulls preserved.


## 8. Review Targeted Categories After Transformation

The objective is to make the new business states visible without inventing information.

In [13]:
for col in ["Offer", "Internet Type", "Churn Category", "Churn Reason"]:
    print(f"\n{col}")
    print(df_clean[col].value_counts(dropna=False))


Offer
Offer
No Offer    3877
Offer B      824
Offer E      805
Offer D      602
Offer A      520
Offer C      415
Name: count, dtype: int64

Internet Type
Internet Type
Fiber Optic            3035
DSL                    1652
No Internet Service    1526
Cable                   830
Name: count, dtype: int64

Churn Category
Churn Category
NaN                5174
Competitor          841
Attitude            314
Dissatisfaction     303
Price               211
Other               200
Name: count, dtype: int64

Churn Reason
Churn Reason
NaN                                          5174
Competitor had better devices                 313
Competitor made better offer                  311
Attitude of support person                    220
Don't know                                    130
Competitor offered more data                  117
Competitor offered higher download speeds     100
Attitude of service provider                   94
Price too high                                 78
Product dissat

## 9. Notebook 2 Conclusion

Text and categorical standardization is complete. Only approved semantic transformations have
been applied, while structural churn nulls remain intact.

**Next:** `03_customer_flags_and_integrity.ipynb`